# Jour 1 · Comprendre le signal et découper le temps


## Objectifs

- distinguer tendance, saisonnalité et bruit de façon intuitive
- lisser un signal avec une moyenne mobile
- séparer passé et futur sans fuite temporelle

Jusqu'ici, nous avons surtout rendu les données fiables. Nous allons maintenant apprendre à **décrire leur comportement** avant de chercher à prévoir ou à détecter une anomalie.

## Préparer une vue horaire

Le fichier préparé contient une mesure toutes les 15 minutes. Pour observer plusieurs semaines sans surcharger le graphique, nous calculons ici une moyenne par heure.

`select_dtypes("number")` conserve uniquement les colonnes numériques : on ne peut pas calculer la moyenne d'un identifiant comme `hvac_01`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


plt.style.use("seaborn-v0_8-whitegrid")
df = pd.read_csv(DATA_DIR / "prepared" / "iot_hvac_clean.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.set_index("timestamp").sort_index()
hourly = df.select_dtypes("number").resample("1h").mean()
hourly.head()

## Trois mots pour décrire un signal

- **Tendance** : évolution lente du niveau général. Exemple : la vibration augmente progressivement pendant plusieurs semaines à cause d'une usure.
- **Saisonnalité** : motif qui revient à intervalle régulier. Exemple : la puissance monte chaque jour ouvré quand le bâtiment est occupé.
- **Bruit** : petites variations rapides qui restent après les motifs explicables. Il peut venir du capteur, de l'environnement ou du procédé.

```text
valeur observée ≈ niveau général + motifs répétés + variations résiduelles
```

Cette écriture est une grille de lecture, pas une séparation parfaite. Un pic inhabituel n'est pas automatiquement du bruit, et un changement répété n'est saisonnier que si sa périodicité est suffisamment stable.

### Questions à poser devant un graphique

1. Le niveau général monte-t-il ou descend-il ?
2. Un motif revient-il chaque jour ou chaque semaine ?
3. L'amplitude des variations change-t-elle ?
4. Certains écarts sont-ils isolés ou persistants ?

## La moyenne mobile

Une moyenne mobile remplace chaque point par la moyenne d'une fenêtre de points voisins. Sur une série horaire :

- une fenêtre de `24` points représente environ 24 heures ;
- une fenêtre de `24 * 7` représente environ une semaine ;
- une grande fenêtre lisse davantage, mais réagit plus lentement aux changements.

Une moyenne mobile ne prédit pas le futur : elle aide d'abord à voir le niveau local.

![Tendance, saisonnalité, bruit et signal observé présentés sur quatre graphiques](../assets/jour_01/03_tendance_saisonnalite_bruit.png)

*Le signal observé combine plusieurs phénomènes ; ces mots servent à décrire ce que l'on voit avant de modéliser.*

![Température horaire comparée à des moyennes mobiles de 24 heures et 7 jours](../assets/jour_01/03_moyennes_mobiles.png)

*Une fenêtre longue révèle le niveau lent, mais elle atténue aussi les cycles et les pics.*

In [ ]:
temperature = hourly["temperature_c"]
smooth_24h = temperature.rolling(24, center=True, min_periods=12).mean()
smooth_7d = temperature.rolling(24 * 7, center=True, min_periods=24).mean()

ax = temperature.plot(figsize=(13, 5), alpha=0.35, label="mesure horaire")
smooth_24h.plot(ax=ax, label="moyenne mobile 24 h", linewidth=2)
smooth_7d.plot(ax=ax, label="moyenne mobile 7 jours", linewidth=3)
ax.set_ylabel("°C")
ax.set_title("Du signal brut à sa tendance")
ax.legend()
plt.show()

## Lire le graphique de lissage

La courbe horaire claire conserve les variations rapides. La moyenne sur 24 heures atténue le cycle au sein d'une journée. La moyenne sur sept jours fait surtout ressortir les évolutions lentes.

Ici, `center=True` place la moyenne au centre de la fenêtre : le graphique est plus facile à lire, mais la valeur calculée utilise alors des points situés avant **et après** l'instant affiché. C'est acceptable pour une analyse descriptive du passé, mais interdit pour fabriquer une variable destinée à une prévision en temps réel.

Autre effet à connaître : les bords contiennent moins de voisins. `min_periods` indique le nombre minimal de valeurs nécessaires avant d'afficher une moyenne.

## Faire apparaître une saisonnalité journalière

Pour répondre à la question « que se passe-t-il habituellement à chaque heure ? », nous regroupons toutes les observations qui ont la même heure : toutes les valeurs de 8 h ensemble, toutes celles de 9 h ensemble, etc. La moyenne de chaque groupe forme un **profil journalier moyen**.

Ce profil résume plusieurs jours. Il ne montre ni la dispersion ni les journées exceptionnelles : il faut donc toujours revenir aux données originales avant de conclure.

![Profils journaliers moyens de la température et de la puissance](../assets/jour_01/03_profil_journalier.png)

*Regrouper les mesures par heure fait apparaître le cycle quotidien, notamment la hausse de puissance pendant l'activité du bâtiment.*

In [ ]:
daily_profile = hourly.groupby(hourly.index.hour)[
    ["temperature_c", "power_kw"]
].mean()
daily_profile.index.name = "heure de la journée (UTC)"
daily_profile.plot(subplots=True, figsize=(11, 6), marker="o")
plt.tight_layout()
plt.show()

### À vous de jouer — interpréter le profil journalier

Observez les courbes de température et de puissance puis répondez sans nouveau calcul :

- À quelles heures la puissance semble-t-elle généralement augmenter ?
- Le profil suggère-t-il un lien avec l'occupation du bâtiment ?
- Pourquoi ce graphique ne suffit-il pas à déclarer une anomalie pour une journée précise ?

In [ ]:
# Notez votre interprétation en français.
pass

### À vous de jouer — lisser la puissance

Ajoutez à la puissance horaire une moyenne mobile de 6 heures et une moyenne mobile de 24 heures. Que fait la fenêtre la plus longue ?

**Indice :** Utilisez rolling(6).mean() et rolling(24).mean().

In [ ]:
# power = hourly["power_kw"]
# Calculez power.rolling(...).mean() avec deux fenêtres.
# Tracez les trois courbes sur les dix derniers jours.
pass

## Le découpage temporel

Pour simuler une vraie prévision, l'entraînement ne doit voir que le passé. Un mélange aléatoire laisserait des observations futures influencer le modèle : c'est une **fuite de données**.

```text
passé connu                         futur simulé
├──────── entraînement ─────────────┼──── test ────┤
                                   frontière
```

- **Entraînement** : période utilisée pour apprendre ou régler une méthode.
- **Test** : période plus récente gardée de côté pour simuler son utilisation future.
- **Frontière** : instant après lequel aucune information ne doit servir à l'apprentissage.

Un découpage aléatoire convient à certaines données indépendantes, mais pas ici : il donnerait au modèle des morceaux du futur tout en lui demandant de prédire des morceaux du passé.

La fuite peut aussi être plus discrète : calculer une normalisation sur toute la période, interpoler à travers la frontière ou utiliser une moyenne mobile centrée comme variable prédictive permettrait au passé de profiter d'informations futures.

![Découpage chronologique entre entraînement et test opposé à un mélange aléatoire](../assets/jour_01/03_decoupage_train_test.png)

*Le test doit se situer après l'entraînement afin de reproduire une utilisation réelle : apprendre dans le passé, vérifier dans le futur.*

In [ ]:
cutoff = hourly.index.max() - pd.Timedelta(days=7)
train = hourly.loc[hourly.index <= cutoff]
test = hourly.loc[hourly.index > cutoff]

print("Entraînement :", train.index.min(), "→", train.index.max(), len(train), "points")
print("Test          :", test.index.min(), "→", test.index.max(), len(test), "points")

ax = hourly["temperature_c"].plot(figsize=(13, 4), color="0.7", label="toutes les données")
train["temperature_c"].plot(ax=ax, label="passé / entraînement")
test["temperature_c"].plot(ax=ax, label="futur / test")
ax.axvline(cutoff, color="black", linestyle="--", label="frontière")
ax.legend()
plt.show()

### À vous de jouer — vérifier l'absence de fuite

Vérifiez par une assertion que le dernier timestamp d'entraînement est strictement antérieur au premier timestamp de test. Calculez aussi la part du test en pourcentage.

**Indice :** Comparez train.index.max() et test.index.min().

In [ ]:
# assert train.index.____() < test.index.____()
# test_share = 100 * len(____) / len(____)
pass

### À vous de jouer — repérer les fuites temporelles

Dites si chaque pratique est correcte ou provoque une fuite :

1. Trier les dates puis prendre les sept derniers jours comme test.
2. Mélanger toutes les lignes avant de créer le train et le test.
3. Calculer la moyenne et l'écart-type uniquement sur le train.
4. Remplir une valeur du train avec une mesure située dans le test.
5. Examiner le score final sur le test après avoir choisi le modèle sur le train.

In [ ]:
# Notez « correct » ou « fuite » et justifiez chaque réponse.
pass

## À retenir

Le lissage aide à lire le signal, mais peut cacher les pics. Le découpage doit toujours respecter la flèche du temps : le passé sert à apprendre, le futur sert à vérifier.

### Mini-mémo

| Besoin | Commande / idée |
|---|---|
| Lisser sur 24 points | `serie.rolling(24).mean()` |
| Profil moyen par heure | `groupby(index.hour).mean()` |
| Garder les dates avant une frontière | `df.loc[df.index <= cutoff]` |
| Garder les dates après une frontière | `df.loc[df.index > cutoff]` |
| Vérifier l'ordre | `train.index.max() < test.index.min()` |